In [2]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

print("YOLO loaded successfully!")

YOLO loaded successfully!


In [3]:
results = model("test.jpg", save=True)

print("Detection completed!")


WARNING Image Read Error c:\CROWD_CONTROL_SYSTEM\test.jpg
Results saved to C:\CROWD_CONTROL_SYSTEM\runs\detect\predict-4
Detection completed!


In [4]:
results = model("people.jpg", save=True)

print("Detection completed!")


image 1/1 c:\CROWD_CONTROL_SYSTEM\people.jpg: 448x640 10 persons, 1 motorcycle, 459.2ms
Speed: 17.8ms preprocess, 459.2ms inference, 6.1ms postprocess per image at shape (1, 3, 448, 640)
Results saved to C:\CROWD_CONTROL_SYSTEM\runs\detect\predict-4
Detection completed!


In [5]:
person_count = 0

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0])

        if class_id == 0:
            person_count += 1

print("Number of people detected:", person_count)

Number of people detected: 10


In [6]:
if person_count < 10:
    crowd_level = "LOW"
elif person_count < 20:
    crowd_level = "MEDIUM"
else:
    crowd_level = "HIGH"

print("Crowd Level:", crowd_level)

Crowd Level: MEDIUM


In [7]:
if crowd_level == "LOW":
    print("Status: Crowd is under control.")
elif crowd_level == "MEDIUM":
    print("Status: Moderate crowd detected. Monitor the area.")
else:
    print("Status: High crowd detected. Immediate crowd control required.")

Status: Moderate crowd detected. Monitor the area.


In [8]:
results = model("crowd2.jpg", save=True)

person_count = 0

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0])

        if class_id == 0:
            person_count += 1

print("Number of people detected:", person_count)


image 1/1 c:\CROWD_CONTROL_SYSTEM\crowd2.jpg: 448x640 9 persons, 217.2ms
Speed: 4.3ms preprocess, 217.2ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to C:\CROWD_CONTROL_SYSTEM\runs\detect\predict-4
Number of people detected: 9


In [9]:
from ultralytics import YOLO
import cv2

model = YOLO("yolo26n.pt")

SAFE_LIMIT = 10

results = model("people.jpg")

person_count = 0
for box in results[0].boxes:
    class_id = int(box.cls[0])
    class_name = model.names[class_id]
    if class_name == "person":
        person_count += 1

print(f"People detected: {person_count}")

if person_count > SAFE_LIMIT:
    print("⚠️ ALERT: Area is OVERCROWDED!")
else:
    print("✅ Crowd level is normal.")

annotated_frame = results[0].plot()
cv2.putText(annotated_frame, f"Count: {person_count}", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

cv2.imwrite("output_with_count.jpg", annotated_frame)


image 1/1 c:\CROWD_CONTROL_SYSTEM\people.jpg: 448x640 10 persons, 1 motorcycle, 213.0ms
Speed: 5.3ms preprocess, 213.0ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)
People detected: 10
✅ Crowd level is normal.


True

In [10]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")  # small model, better than nano for small objects

results = model("crowd2.jpg", imgsz=1920, conf=0.1)


image 1/1 c:\CROWD_CONTROL_SYSTEM\crowd2.jpg: 1280x1920 103 persons, 1 car, 1 truck, 1 backpack, 2 umbrellas, 3 handbags, 3012.8ms
Speed: 53.7ms preprocess, 3012.8ms inference, 0.7ms postprocess per image at shape (1, 3, 1280, 1920)


In [11]:
annotated = results[0].plot()
import cv2
cv2.imwrite("crowd2_detected.jpg", annotated)

True

In [12]:
person_count = sum(1 for cls in results[0].boxes.cls if model.names[int(cls)] == "person")
print(f"People detected: {person_count}")

SAFE_LIMIT = 50
if person_count > SAFE_LIMIT:
    print("⚠️ ALERT: Area is OVERCROWDED!")
else:
    print("✅ Crowd level is normal.")

People detected: 103
⚠️ ALERT: Area is OVERCROWDED!


In [13]:
pip install sahi --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [14]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolo26s.pt",
    confidence_threshold=0.1,
    device="cpu"
)

result = get_sliced_prediction(
    "crowd2.jpg",
    detection_model,
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

person_count_sahi = sum(1 for obj in result.object_prediction_list if obj.category.name == "person")
print(f"People detected (SAHI): {person_count_sahi}")

result.export_visuals(export_dir="sahi_output/")

Performing prediction on 12 slices.
People detected (SAHI): 124


In [15]:
result = get_sliced_prediction(
    "crowd2.jpg",
    detection_model,
    slice_height=256,
    slice_width=256,
    overlap_height_ratio=0.3,
    overlap_width_ratio=0.3,
)

Performing prediction on 54 slices.


In [16]:
person_count_sahi

124

In [17]:
person_count_sahi_256 = sum(1 for obj in result.object_prediction_list if obj.category.name == "person")
print(f"People detected (SAHI, 256px tiles): {person_count_sahi_256}")

People detected (SAHI, 256px tiles): 235


In [18]:
result = get_sliced_prediction(
    "crowd2.jpg",
    detection_model,
    slice_height=128,
    slice_width=128,
    overlap_height_ratio=0.3,
    overlap_width_ratio=0.3,
)

Performing prediction on 187 slices.


In [19]:
result.export_visuals(export_dir="sahi_output_256/")

In [20]:
person_count_sahi_128 = sum(1 for obj in result.object_prediction_list if obj.category.name == "person")
print(f"People detected (SAHI, 128px tiles): {person_count_sahi_128}")

People detected (SAHI, 128px tiles): 453


In [ ]:
result = get_sliced_prediction(
    "crowd2.jpg",
    detection_model,
    slice_height=80,
    slice_width=80,
    overlap_height_ratio=0.3,
    overlap_width_ratio=0.3,
)

Performing prediction on 486 slices.


In [22]:
result.export_visuals(export_dir="sahi_output_120/")

In [23]:
person_count_sahi_120 = sum(1 for obj in result.object_prediction_list if obj.category.name == "person")
print(f"People detected (SAHI, 120px tiles): {person_count_sahi_120}")

People detected (SAHI, 120px tiles): 468
